# Quantization Aware Training + Knowledge Distillation Benchmarking

In [ ]:
import os
import torch
import torch.onnx
from torch.ao.quantization.quantize_fx import convert_fx
import torchvision

from model_compression.src.utils import load_data
from model_compression.src.quantization.utils.model_setup import setup_qat_student_model, quantization_mode
# from model_compression.src.quantization.utils.conversions.onnx import export_pytorch_to_onnx
from model_compression.src.utils import benchmark
from model_compression.src.utils.model_setup import setup_model
from model_compression.src.utils import test_inference, test_inference_onnx
from model_compression.src.quantization.core import quantize_pytorch_model

### Load Original and Quantized model

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# pretrained_weights = f"models/SkinCancer/Quantized/quantized_student_state.pth"
# batch_size = 32
# dataloaders = load_data(dataset="SkinCancer", batch_size=batch_size)
# num_classes = len(dataloaders["train"].dataset.classes)
# model = setup_qat_student_model(model_name="mobilenet_v2",num_classes=num_classes)

# example_inputs = next(iter(dataloaders["train"]))[0].to(device)
# student_model = quantization_mode(model, "fx", example_inputs=example_inputs)

# # Move the model to CPU if needed (conversion is typically done on CPU).
# student_model = student_model.to("cpu")

# quantized_model = convert_fx(student_model)

# # Now load the state dict.
# state_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")
# quantized_model.load_state_dict(state_dict)

# # Set to eval mode.
# quantized_model.eval()

# teacher_model = setup_model("mobilenet_v2", None, num_classes)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pretrained_weights = f"models/SkinCancer/Quantized/mobilenet_v2_qat_kd.pth"
batch_size = 32
dataloaders = load_data(dataset="SkinCancer", batch_size=batch_size)
num_classes = len(dataloaders["train"].dataset.classes)
model = setup_model(model_name="mobilenet_v2", pretrained_weights=None, num_classes=num_classes)

example_inputs = next(iter(dataloaders["train"]))[0].to("cpu")
# exported_model = capture_pre_autograd_graph(model, (example_inputs,))

student_model = quantization_mode(model, "export", example_inputs=(example_inputs,)).to(device)

# quantized_model = quantize_pytorch_export_model(student_model, None)

# Now load the state dict.
state_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")
student_model.load_state_dict(state_dict)

quantized_model = quantize_pytorch_model(student_model, None)

# # Set to eval mode.
# torch.ao.quantization.move_exported_model_to_eval(quantized_model)

teacher_model = setup_model("mobilenet_v2", "models/SkinCancer/mobilenet_v2_best_model.pth", num_classes).to(device)

### Perform Benchmarking (model_size, inference time, throughput, memory usage)

In [ ]:
# device = torch.device("cpu")
test_inference(quantized_model, dataloaders["test"], device, None)

In [ ]:
# device = torch.device("cpu")

benchmark(model1=teacher_model, model2=quantized_model, dataloader=dataloaders["test"], device=device)

In [ ]:
print(os.path.getsize("quantized_student.onnx") / 1e6)